In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from timm.models import create_model
from datasets import build_dataset
from compression.int_approxi_func import *

/home/chenzhiqiang/miniconda3/envs/det/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import argparse

parser = argparse.ArgumentParser()
parser.add_argument('--data-path', default='/home/chenzhiqiang/datasets/ImageNet', type=str,
                    help='dataset path')
parser.add_argument('--data-set', default='IMNET', choices=['CIFAR', 'IMNET', 'INAT', 'INAT19'],
                    type=str, help='Image Net dataset path')
parser.add_argument('--batch-size', default=64, type=int)
parser.add_argument('--num_workers', default=8, type=int)
parser.add_argument('--input-size', default=224, type=int, help='images input size')
parser.add_argument('--color-jitter', type=float, default=0.3, metavar='PCT',
                    help='Color jitter factor (default: 0.3)')
parser.add_argument('--aa', type=str, default='rand-m9-mstd0.5-inc1', metavar='NAME',
                    help='Use AutoAugment policy. "v0" or "original". " + \
                            "(default: rand-m9-mstd0.5-inc1)'),
parser.add_argument('--smoothing', type=float, default=0.1, help='Label smoothing (default: 0.1)')
parser.add_argument('--train-interpolation', type=str, default='bicubic',
                    help='Training interpolation (random, bilinear, bicubic default: "bicubic")')
parser.add_argument('--reprob', type=float, default=0.25, metavar='PCT',
                    help='Random erase prob (default: 0.25)')
parser.add_argument('--remode', type=str, default='pixel',
                    help='Random erase mode (default: "pixel")')
parser.add_argument('--recount', type=int, default=1,
                    help='Random erase count (default: 1)')
parser.add_argument('--resplit', action='store_true', default=False,
                    help='Do not random erase first (clean) augmentation split')
    
args = parser.parse_args(['--data-set', 'IMNET'])

In [3]:
model = create_model('deit_small_distilled_patch16_224', pretrained=True)
datasets, nb_classes = build_dataset(is_train=True, args=args)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
datasets, nb_classes = build_dataset(is_train=False, args=args) # Use val set for evaluation
    
data_loader = torch.utils.data.DataLoader(
    datasets, 
    batch_size=args.batch_size,
    num_workers=args.num_workers,
    pin_memory=True,
    shuffle=True
)

data_iter = iter(data_loader)
images, targets = next(data_iter)
images = images[:32].to(device)
model.to(device)

/home/chenzhiqiang/miniconda3/envs/det/lib/python3.8/site-packages/torchvision/transforms/transforms.py:287: UserWarning: Argument interpolation should be of type InterpolationMode instead of int. Please, use InterpolationMode enum.
  warnings.warn(


VisionTransformerDistilled(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU()
        (drop1): Dropout(p=0.0, inplace=False)
       

## 1. Eval Cosine Similarity

In [4]:
from compression.utils import ActivationCaptureHook, register_eval_hooks

hook_dict, handles = register_eval_hooks(model)

print("Running FP model forward pass to capture activations...")
with torch.no_grad():
    _ = model(images)

for handle in handles:
    handle.remove()

Running FP model forward pass to capture activations...


In [12]:
print("\nEvaluating Cosine Similarity between FP and INT approximated modules...")
print("-" * 80)
print(f"{'Layer Name':<45} | {'Module Type':<15} | {'Cosine Sim':<10}")
print("-" * 80)

gelu_results = []
softmax_results = []
layernorm_results = []

# 5. Evaluate approximations and compute Cosine Similarity
for name, info in hook_dict.items():
    mod_type = info['type']
    

    
    # Instantiate corresponding Integer Approximation Module
    if mod_type == 'Softmax':
        fp_input = info['inputs'][0].to(device)
        fp_output = info['outputs'][0].to(device)
        int_mod = IntSoftmaxFixed().to(device)
        mod_name_str = "Softmax"
        result_list = softmax_results
        
    else:
        hook = info['hook']
        orig_module = info['orig_module']
        fp_input = hook.inputs[0].to(device)
        fp_output = hook.outputs[0].to(device)
        mod_name_str = mod_type.__name__
    
        if isinstance(orig_module, nn.GELU):
            int_mod = IntGeLU_LUT().to(device)
            result_list = gelu_results
            
        elif isinstance(orig_module, nn.LayerNorm):
            # For LayerNorm, we MUST copy the trained weights and biases from the original FP module
            int_mod = IntLayerNorm(orig_module.normalized_shape, orig_module.eps).to(device)
            int_mod.weight.data = orig_module.weight.data.clone()
            int_mod.bias.data = orig_module.bias.data.clone()
            result_list = layernorm_results
        
    int_mod.eval()
    
    # 6. Forward pass through the Integer Module using the captured FP input
    with torch.no_grad():
        int_output = int_mod(fp_input)
        
    # 7. Calculate Cosine Similarity
    # Flatten all dimensions except the Batch dimension (dim=0) to calculate similarity per sample
    fp_flat = fp_output.view(fp_output.size(0), -1)
    int_flat = int_output.view(int_output.size(0), -1)
    
    # F.cosine_similarity returns shape [Batch_size]. We take the mean across the batch.
    cos_sim = F.cosine_similarity(fp_flat, int_flat, dim=1).mean().item()
    
    result_list.append((name, cos_sim))
    
    # print(f"{name:<45} | {mod_type.__name__:<15} | {cos_sim:.6f}")

print("\nEvaluating Cosine Similarity between FP and INT approximated modules...")

gelu_results = []
softmax_results = []
layernorm_results = []

# 5. Evaluate approximations and compute Cosine Similarity
for name, info in hook_dict.items():
    mod_type = info['type']
    

    
    # Instantiate corresponding Integer Approximation Module
    if mod_type == 'Softmax':
        fp_input = info['inputs'][0].to(device)
        fp_output = info['outputs'][0].to(device)
        int_mod = IntSoftmaxFixed().to(device)
        mod_name_str = "Softmax"
        result_list = softmax_results
        
    else:
        hook = info['hook']
        orig_module = info['orig_module']
        fp_input = hook.inputs[0].to(device)
        fp_output = hook.outputs[0].to(device)
        mod_name_str = mod_type.__name__
    
        if isinstance(orig_module, nn.GELU):
            int_mod = IntGeLU_LUT().to(device)
            result_list = gelu_results
            
        elif isinstance(orig_module, nn.LayerNorm):
            # For LayerNorm, we MUST copy the trained weights and biases from the original FP module
            int_mod = IntLayerNorm(orig_module.normalized_shape, orig_module.eps).to(device)
            int_mod.weight.data = orig_module.weight.data.clone()
            int_mod.bias.data = orig_module.bias.data.clone()
            result_list = layernorm_results
        
    int_mod.eval()
    
    # 6. Forward pass through the Integer Module using the captured FP input
    with torch.no_grad():
        int_output = int_mod(fp_input)
        
    # 7. Calculate Cosine Similarity
    # Flatten all dimensions except the Batch dimension (dim=0) to calculate similarity per sample
    fp_flat = fp_output.view(fp_output.size(0), -1)
    int_flat = int_output.view(int_output.size(0), -1)
    
    # F.cosine_similarity returns shape [Batch_size]. We take the mean across the batch.
    cos_sim = F.cosine_similarity(fp_flat, int_flat, dim=1).mean().item()
    
    result_list.append((name, cos_sim))
    
    # print(f"{name:<45} | {mod_type.__name__:<15} | {cos_sim:.6f}")

print("-" * 80)
def print_section(title, results):
    if not results:
        return
    print(f"\n{title}")
    print(f"{'Layer Name':<45} | {'Cosine Sim':<10}")
    print("-" * 60)
    sims = []
    for name, cos_sim in results:
        print(f"{name:<45} | {cos_sim:.6f}")
        sims.append(cos_sim)
    avg_sim = sum(sims) / len(sims)
    # print(f"Average: {avg_sim:.6f}")
    print(f"{'Average':<45} | {avg_sim:.6f}")
    print("-" * 60)

print_section("GELU Modules:", gelu_results)
print_section("Softmax Modules:", softmax_results)
print_section("LayerNorm Modules:", layernorm_results)

print("-" * 80)
print("Evaluation completed.")


Evaluating Cosine Similarity between FP and INT approximated modules...
--------------------------------------------------------------------------------
Layer Name                                    | Module Type     | Cosine Sim
--------------------------------------------------------------------------------

Evaluating Cosine Similarity between FP and INT approximated modules...
--------------------------------------------------------------------------------

GELU Modules:
Layer Name                                    | Cosine Sim
------------------------------------------------------------
blocks.0.mlp.act                              | 0.999752
blocks.1.mlp.act                              | 0.999130
blocks.2.mlp.act                              | 0.999116
blocks.3.mlp.act                              | 0.998919
blocks.4.mlp.act                              | 0.998919
blocks.5.mlp.act                              | 0.999089
blocks.6.mlp.act                              | 0.999149


# 2. Eval IN-1K Classification Accuracy with Non-Linear replaced